> 업로드 정리 주석(2026-10-30)
> - 이 노트북은 제공된 코드·해석의 기록이며 이번 정리에서 실제 데이터를 넣어 재실행하지 않았습니다.
> - 현재 코드의 분석 상한은 예측기준월 202602입니다. 아래 해석에 남은 2026.03·2026.02 원자료 등의 옛 수치는 현재 코드 재실행 결과로 확정하면 안 됩니다.
> - 전체 기간 Spearman 상관은 탐색 결과입니다. 최신 학습 구간 Pearson 분석 및 최종 변수 선택과 구분하세요.
> - 카드 13절의 라벨 분석은 202407~202508을 사용합니다. 최신 학습 범위(상관분석 기준 202301~202502)와 달라 모델 선택 근거로 재사용하기 전에 수정·재검증해야 합니다.
> - 낮은 상관만으로 독립성을 입증하거나 변수를 자동 채택하지 않습니다.
> - 실제 입력 데이터와 실행 출력은 공개 업로드 묶음에서 제외했습니다.


# 03. 카드 + 통신 합치기 EDA
대상: `카드데이터_전처리_v05.csv`(동×월×카드업종, 파생지표 포함) + `통신데이터_전처리_v03.csv`(동×월, 지표 4개)
합치는 열쇠: `기준월` + `행정동코드` (통신 한 줄이 같은 동·달의 카드 11업종 줄에 붙는다)

**규칙**
- 지표는 t = 2023.01 ~ 2026.02까지만 본다. 2026.02 행이 시험 문제다. (카드·통신 모두 t−1월까지 자료만 쓴 값)
- 그 달 자체 인구 값(`거주인구_월` 등)은 모델 입력이 아니므로 가져오지 않는다.
- 폐업 라벨과 연결 짓는 분석은 하지 않는다.
- `점포수_t`가 적은 4개 업종(자동차·의류/잡화·교육·여행/숙박)은 12·15번이 불안정하므로(`쉬운설명_점포수0_원인.md`) 따로 나눠서도 본다.

**구성** 1 합치기와 점검 → 2 빈칸 → 3 상관 → 4 업종 나눠서 상관 → 5 동네 단위 비교 → 6 동네 유형별 → 7 강동동 교차 확인 → 8 정리

> ⚠ **시험 기준월을 2026.03 → 2026.02로 변경** (자료는 2026.01까지, 정답 기간 2026.02~2026.07). 위 설명과 코드(`CUT` 등)는 새 기준으로 고쳤다. **아래 '결과 해석'과 마지막 정리 표의 숫자·그림 설명은 이전 기준(자료 2026.02까지, 시험 2026.03)으로 쓴 것이라 한 달씩 어긋난다. 다시 실행해서 그림과 숫자를 새로 확인한 뒤 갱신한다.**


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False,
                     "figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}".rstrip("0").rstrip("."))   # 8.3e+04 같은 지수 표기 대신 쉼표 숫자로 (소수는 최대 4자리)

CARD, TEL = "카드데이터_전처리_v05.csv", "통신데이터_전처리_v03.csv"   # 이 노트북과 같은 폴더(완전쓰는거_ver2)에서 실행
TMAX = 202602
CIND = ["소비추세_6개월", "소비전년비", "소비변동계수_12개월", "점포당소비_z", "소비점포증가율격차"]
TIND = ["직장인구_6개월평균", "방문인구_시간대변동계수_6개월", "방문인구_전년대비증감률", "방문인구비중"]
ALL = CIND + ["점포수_t"] + TIND
SMALL4 = ["자동차", "의류/잡화", "교육", "여행/숙박"]          # 점포 수가 적어 12·15번이 불안정한 업종

## 1. 합치기와 점검

In [ ]:
card = pd.read_csv(CARD, encoding="utf-8-sig"); card = card[card.기준월 <= TMAX]
tel = pd.read_csv(TEL, encoding="utf-8-sig"); tel = tel[tel.기준월 <= TMAX]
m = card.merge(tel[["기준월", "행정동코드"] + TIND], on=["기준월", "행정동코드"], how="left", validate="m:1")
m["월"] = pd.to_datetime(m.기준월.astype(str), format="%Y%m")

print("카드", card.shape, "| 통신", tel.shape, "| 합친 표", m.shape)
print("행 수 유지:", len(m) == len(card), "| 통신 지표 4개가 전부 빈칸인 행:", int(m[TIND].isna().all(axis=1).sum()), "(2023.01~06, 6개월 평균이라 정상)")
print("통신 키 자체 누락(동·월이 통신에 없음):", len(card.merge(tel[['기준월','행정동코드']], on=['기준월','행정동코드'], how='left', indicator=True).query("_merge=='left_only'")))
print("동", m.행정동코드.nunique(), "| 업종", m.카드업종.nunique(), "| 월", m.기준월.nunique())
m[ALL].describe().T.round(3)

### 결과 해석
- 카드 87,945행에 통신 지표 4개를 붙여 **20열**. 행 수가 유지되고, 통신에 없는 동·월은 0건이다. 통신 한 줄이 같은 동·달의 카드 11업종 줄에 붙는다.
- 동 205 · 업종 11 · 월 39.
- **이상한 점:** 없음. (출력의 "통신 키 못 찾은 행 13,530"은 키가 없는 것이 아니라 시작 달 이전이라 지표가 비어 있는 행 수다.)

## 2. 빈칸

In [ ]:
na = m.groupby("월")[ALL].apply(lambda g: g.isna().mean())
fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(na.T, aspect="auto", cmap="Reds", vmin=0, vmax=1); ax.grid(False)
ax.set_yticks(range(len(ALL)), ALL, fontsize=8)
ax.set_xticks(range(0, len(na), 3), [d.strftime("%y.%m") for d in na.index[::3]], rotation=60, fontsize=8)
ax.set_title("지표별 빈칸 비율 (월 × 지표, 진할수록 빈칸 많음)"); plt.colorbar(im); plt.show()

first = {c: int(m.loc[m[c].notna(), "기준월"].min()) for c in ALL}
print("첫 값이 있는 달", first)
test = m[m.기준월 == TMAX]
print("\n2026.02 행 빈칸 수 (총 %d행)" % len(test)); print(test[ALL].isna().sum().to_string())

### 결과 해석
- **빈칸 구조가 명세서의 시작 t와 같다.** 소비추세·점포당소비_z·직장·방문변동계수·방문비중은 2023.06까지, 소비변동계수·증가율 격차는 2023.12까지, 소비전년비·방문전년비는 2024.06까지 비어 있다. 점포수_t는 비는 달이 없다.
- 점포당소비_z와 증가율 격차는 **이후에도 약 7%가 계속 비어 있다**(그림의 연한 줄무늬). 점포 0곳 조합 때문이다.
- 2026.03 행은 점포당소비_z 151행, 소비점포증가율격차 155행만 비어 있고 나머지 지표는 0건이다.
- **이상한 점:** 없음. 의도한 빈칸과 점포 0곳 때문에 생긴 빈칸뿐이다.

In [ ]:
# 학습에 쓸 수 있는 행: 모든 지표가 있는 행 (가장 늦게 시작하는 지표 = 방문 전년비 2024.07)
full = m[ALL].notna().all(axis=1)
by = full.groupby(m.기준월).mean()
print("모든 지표가 있는 행 비율: 2024.07 이후 평균", round(by.loc[202407:].mean(), 3), "| 2026.02", round(by.loc[TMAX], 3))
print("12·15번을 빼고(점포 수 문제) 카드·통신 지표만 보면 2026.02:",
      round(m.loc[m.기준월 == TMAX, ['소비추세_6개월','소비전년비','소비변동계수_12개월'] + TIND].notna().all(axis=1).mean(), 3))
by.loc[202407:].describe().round(3)

### 결과 해석
- **모든 지표가 있는 행 비율:** 2024.07 이후 평균 92.9%, 2026.03은 93.1%. 나머지 약 7%는 점포당소비_z·증가율 격차가 비는 행이다.
- 12·15번을 빼고 카드·통신 지표만 보면 2026.03은 **100%**가 채워져 있다.
- **이상한 점:** 없음. 학습 행의 약 7%가 12·15번 때문에 불완전하다는 점만 기억하면 된다(트리 모델은 빈칸을 그대로 처리).

## 3. 지표끼리 상관 (Spearman, t = 2023.01 ~ 2026.03 전체)

In [ ]:
def heat(cr, title, size=(8, 6.5)):
    fig, ax = plt.subplots(figsize=size)
    im = ax.imshow(cr, vmin=-1, vmax=1, cmap="RdBu_r"); ax.grid(False)
    ax.set_xticks(range(len(cr)), cr.columns, rotation=60, ha="right", fontsize=8); ax.set_yticks(range(len(cr)), cr.index, fontsize=8)
    for i in range(len(cr)):
        for j in range(len(cr)): ax.text(j, i, f"{cr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(title); plt.colorbar(im); plt.tight_layout(); plt.show()

def pairs(cr, th=.8):
    return [(a, b, round(cr.loc[a, b], 2)) for i, a in enumerate(cr.index) for b in cr.columns[i+1:] if abs(cr.loc[a, b]) > th] or "없음"

cr = m[ALL].corr(method="spearman")
heat(cr, "Spearman 상관 (카드 + 통신 지표)")
print("상관 0.8 초과 쌍:", pairs(cr))
print("0.5 초과 쌍:", pairs(cr, .5))

### 결과 해석 (Spearman, 전체 기간)
- **상관 0.8을 넘는 쌍이 없다.** 0.5를 넘는 쌍은 소비변동계수 × 점포수_t −0.55 하나뿐이다(점포가 적을수록 소비가 들쭉날쭉).
- 그 밖에 눈에 띄는 쌍: 소비전년비 × 소비점포증가율격차 0.46, 점포수_t × 직장인구 0.32, 직장인구 × 방문변동계수 0.38, 점포당소비_z × 방문인구비중 −0.32(방문이 많은 동네일수록 점포당 소비가 낮은 경향), 점포당소비_z × 점포수_t · 방문변동계수 각 −0.27.
- 카드 지표와 통신 지표 사이의 상관은 대부분 0.1 안팎이다.
- **이상한 점:** 없음. 하나만 남겨야 할 지표는 없다.

## 4. 점포 수가 충분한 업종과 부족한 4개 업종을 나눠서 상관

In [ ]:
ok_ = m[~m.카드업종.isin(SMALL4)]; sm = m[m.카드업종.isin(SMALL4)]
cr_ok, cr_sm = ok_[ALL].corr(method="spearman"), sm[ALL].corr(method="spearman")
diff = (cr_sm - cr_ok).abs()
print("점포 충분 7개 업종: 0.8 초과", pairs(cr_ok), "| 부족 4개 업종: 0.8 초과", pairs(cr_sm))
d_ = diff.where(np.triu(np.ones(diff.shape, bool), 1)).stack().sort_values(ascending=False).head(6)
print("두 그룹의 상관 차이가 큰 쌍"); print(d_.round(2).to_string())
heat(cr_ok, "점포 충분 7개 업종", (7, 5.8))

### 결과 해석 (업종을 나눠서 본 상관)
- 점포가 충분한 7개 업종과 부족한 4개 업종 모두 **0.8 초과 쌍이 없다.**
- 두 그룹의 상관 차이가 큰 쌍은 소비전년비 × 소비변동계수(차이 0.29), 점포당소비_z × 직장인구(0.18), 소비변동계수 × 점포수_t(0.16) 순이다.
- **이상한 점·해석:** 점포 부족 업종 때문에 지표 간 관계가 크게 달라지지는 않는다. 다만 이 업종은 값 자체가 불안정하다(`쉬운설명_점포수0_원인.md`).

## 5. 동네 단위 비교 — 카드 지표(동 중앙값)와 통신 지표

In [ ]:
dong = m[~m.카드업종.isin(SMALL4)].groupby(["기준월", "행정동코드"]).agg(
    소비추세=("소비추세_6개월", "median"), 소비전년비=("소비전년비", "median"),
    소비변동계수=("소비변동계수_12개월", "median"), **{c: (c, "first") for c in TIND}).reset_index()
cc = dong.drop(columns=["기준월", "행정동코드"]).corr(method="spearman").loc[["소비추세", "소비전년비", "소비변동계수"], TIND]
print("동네 단위 상관: 행 = 카드(동 중앙값, 점포 충분 7업종), 열 = 통신"); print(cc.round(2).to_string())

t = dong[dong.기준월 == TMAX]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(t.방문인구_전년대비증감률, t.소비전년비, s=12, alpha=.6); ax[0].set_xlabel("방문인구 전년대비증감률"); ax[0].set_ylabel("소비전년비 (동 중앙값)")
ax[1].scatter(t.방문인구비중, t.소비변동계수, s=12, alpha=.6); ax[1].set_xlabel("방문인구비중"); ax[1].set_ylabel("소비변동계수 (동 중앙값)")
fig.suptitle("2026.02 동네별"); plt.tight_layout(); plt.show()

### 결과 해석 (동 단위로 합친 카드 지표 × 통신 지표)
- 카드 지표(점포 충분 7업종의 동 중앙값)와 통신 지표의 상관은 **대부분 0.13 이하**다. 두 데이터가 서로 거의 독립적인 정보를 준다는 뜻이다.
- 예외는 **소비변동계수 × 방문인구비중 0.42**: 방문이 많은 동네일수록 소비가 들쭉날쭉하다.
- 산점도(2026.03): 방문 전년비와 소비 전년비는 **강동동 한 점만 오른쪽 위로 멀리 떨어져 있고(방문 +1.16, 소비 +1.7) 나머지는 0 근처에 뭉쳐 있다.** 방문비중이 클수록 소비변동계수가 위로 흩어진다.
- **이상한 점:** 강동동 한 점이 그림 축을 늘려 나머지가 뭉쳐 보인다. 강동동을 빼고 다시 보는 것이 좋다.

## 6. 동네 유형별 — 방문인구비중으로 나눈 3그룹의 카드 지표

In [ ]:
tt = m[(m.기준월 == TMAX) & (~m.카드업종.isin(SMALL4))].copy()
dv = tt.drop_duplicates("행정동코드").set_index("행정동코드").방문인구비중
typ = pd.qcut(dv, 3, labels=["주거형(방문 적음)", "혼합형", "상권형(방문 많음)"])
tt["유형"] = tt.행정동코드.map(typ)
print("동 수", typ.value_counts().to_dict())
res = tt.groupby("유형", observed=True)[CIND[:3] + ["직장인구_6개월평균", "방문인구_전년대비증감률"]].median().round(3)
res.T.plot.bar(figsize=(10, 3.8), title="동네 유형별 중앙값 (2026.02, 점포 충분 7업종)"); plt.xticks(rotation=20); plt.show()
res

### 결과 해석 (방문인구비중으로 나눈 3그룹, 각 약 68동)
- 소비변동계수의 중앙값은 **주거형 0.073 · 혼합형 0.078 · 상권형 0.089**로 상권형이 높다. 소비추세(0.003~0.004)와 소비전년비(0.045~0.056)는 그룹 사이 차이가 작다.
- 직장인구_6개월평균은 상권형 약 1,333명, 주거형 약 1,135명, 혼합형 약 1,059명. 방문 전년비는 혼합형 0.058, 주거형 0.030, 상권형 0.026.
- **이상한 점:** 그림에서는 직장인구(1,000대)의 막대만 보이고 소비지표(0.0x)는 막대가 보이지 않는다. 지표별로 그림을 나눠야 읽을 수 있다. 위 값은 표에서 읽었다.

## 7. 강동동 교차 확인 (카드와 통신 둘 다 극단값이었던 동)

In [ ]:
g = m[(m.읍면동 == "강동동") & (m.기준월 == TMAX)]
print("통신 지표 (동 전체)"); print(g[TIND].iloc[0].round(3).to_string())
print("\n카드 지표 (업종별)"); print(g[["카드업종", "평균금액", "소비추세_6개월", "소비전년비", "점포수_t"]].round(3).to_string(index=False))
raw_ = card[(card.읍면동 == "강동동")].groupby("기준월").평균금액.sum() / 1e6
print("\n강동동 카드 일평균금액 합계 (백만 원)"); raw_.loc[[202301, 202306, 202312, 202406, 202412, 202503, 202506, 202509 if 202509 in raw_.index else 202506, 202601, 202602]].round(1)

### 결과 해석 (강동동)
- 카드 일평균금액 합계: 2023.01 약 0.86억 → 2024.06 0.95억 → **2024.12 1.64억 → 2025.06 2.16억 → 2025.09 3.25억 → 2026.03 4.51억(5배)**. 2024.12부터 급증한다.
- 2026.03 기준 카드 11개 업종 모두 소비전년비가 +100% 이상이다(자동차 +600%, 교육 +323%, 여행/숙박 +276%, 미용 +213% 등). 통신 방문전년비도 +116%다.
- 점포 수는 음식/주점 106곳, 음식료품 76곳, 유통 56곳, 의료 19곳이고 나머지 업종은 0~9곳이다.
- **이상한 점·해석:** 한 업종이 아니라 **동네 전체가 커졌다.** 개발·입주 같은 실제 변화일 가능성이 있다. 이 동네는 모델에서 극단값이 되므로 확인이 필요하다.

## 8. 정리 — 이상한 점과 이유

각 셀 아래 '결과 해석'에 그림을 보고 읽은 내용을 적었다. 이 표는 핵심 수치만 모은 것이다.

| 항목 | 확인한 점 | 이유·해석 |
|---|---|---|
| 합치기 | 카드 87,945행에 통신 지표 4개를 붙여 20열. 행 수 유지, 통신에 없는 동·월 0건 | `기준월 + 행정동코드`로 1:11(통신 1줄이 카드 11업종에 붙음) 합치기가 문제없다 |
| 빈칸 (2026.03) | 점포당소비_z 151행, 소비점포증가율격차 155행만 비어 있고 나머지 지표는 0건 | 모두 점포 0곳 조합 때문 (`쉬운설명_점포수0_원인.md`) |
| 학습 가능한 행 | 2024.07 이후 모든 지표가 있는 행 비율 평균 92.9% (2026.03은 93.1%). 12·15번을 빼면 2026.03은 100% | 12·15번이 빈칸의 거의 전부 |
| 상관 | 0.8 초과 쌍 없음. 0.5 초과는 소비변동계수 × 점포수 −0.55 하나 | 하나만 남겨야 할 지표 없음. 점포가 적을수록 변동계수가 커진다 |
| 업종 나눠서 상관 | 점포 충분 7업종·부족 4업종 모두 0.8 초과 없음. 상관 차이가 가장 큰 쌍은 소비전년비 × 소비변동계수 (0.29) | 부족 업종 때문에 상관 구조가 크게 달라지지는 않는다 |
| 카드와 통신의 관계 | 동네 단위로 카드 지표(동 중앙값)와 통신 지표의 상관은 대부분 0.13 이하. 예외는 소비변동계수 × 방문인구비중 0.42 | 두 데이터는 서로 다른 정보를 준다. 방문이 많은 동네일수록 소비가 들쭉날쭉하다 |
| 동네 유형 | 방문비중 3그룹(주거형·혼합형·상권형, 각 약 68동)의 소비변동계수 중앙값 0.073 / 0.078 / 0.089. 소비추세·전년비 차이는 작다 | 상권형 동네는 소비 변동이 크다 |
| 강동동 교차 확인 | 카드 일평균금액 합계가 2023.01 약 0.86억 → 2026.03 약 4.5억(5배). 2024.12부터 급증. 11개 업종 모두 전년비 +100% 이상, 통신 방문전년비 +116% | 한 업종이 아니라 동네 전체가 커졌다. 개발·입주 같은 실제 변화일 가능성이 있어 확인 필요. 이 동네는 모델에서 극단값이 된다 |

## 9. 통합 데이터 저장
카드 v05(15열) + 통신 지표 4개를 `기준월 + 행정동코드`로 합친 표를 CSV로 저장한다. (t = 2023.01 ~ TMAX, 보조 열 `월`은 뺀다)

In [ ]:
OUT = "카드통신_통합데이터_v01.csv"
m.drop(columns="월").to_csv(OUT, index=False, encoding="utf-8-sig")
chk = pd.read_csv(OUT, encoding="utf-8-sig")
print(OUT, chk.shape, "| 기준월", chk.기준월.min(), "~", chk.기준월.max())
chk.head(3)